# RAF-DB: py-feat vs Graphormer-lite inference test

Notebook compares `py-feat` against trained Graphormer-lite checkpoints on the same RAF-DB images.
It samples 5 test images per emotion, runs `py-feat` on raw images, runs Graphormer-lite on the corresponding MediaPipe landmarks, and saves comparison tables.

Key outputs:
- `median_probabilities_by_emotion.csv`: median/mean probabilities for `py-feat`, `gformer_s`, `gformer_m`.
- `median_absdiff_by_emotion.csv`: median absolute probability difference vs `py-feat`.
- `class_level_comparison.csv`: per-class top-label accuracy and probability drift.
- `pyfeat_graphormer_predictions.csv`: per-image full probabilities.

In [ ]:

# Same dependency pins used in the RAF-DB Graphormer-lite notebooks.
# Run in Colab/local notebook if the environment is not prepared yet.
%pip install -q "uv>=0.4.0"
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "uv", "pip", "install", "--system", "--quiet", 
    "numpy==1.23.5",
    "pandas==2.0.3",
    "scikit-learn==1.3.2",
    "torch==2.0.0",
    "torchvision==0.15.1",
    "opencv-python-headless==4.7.0.72",
    "mediapipe==0.10.18",
    "py-feat==0.6.2",
    "matplotlib==3.7.5",
    "tqdm==4.66.5",
])

## Configuration

Paths are set for this local project. Change `PROJECT_ROOT` if the notebook is moved.

In [1]:

from pathlib import Path

PROJECT_ROOT = Path('/Users/pelmeshek1706/Desktop/projects/airest-face')
EXPERIMENT_ROOT = PROJECT_ROOT / 'output/jupyter-notebook/emotional_expressivity'
RUN_ROOT = EXPERIMENT_ROOT / 'rafdb_graphormer_lite_full_run_exp'
DATA_CACHE = EXPERIMENT_ROOT / 'rafdb_mediapipe_768_publish/cache'
RESULTS_DIR = EXPERIMENT_ROOT / 'mediapipe_pyfeat_test/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('EXPERIMENT_ROOT:', EXPERIMENT_ROOT)
print('RUN_ROOT:', RUN_ROOT)
print('DATA_CACHE:', DATA_CACHE)
print('RESULTS_DIR:', RESULTS_DIR)


PROJECT_ROOT: /Users/pelmeshek1706/Desktop/projects/airest-face
RUN_ROOT: /Users/pelmeshek1706/Desktop/projects/airest-face/output/jupyter-notebook/rafdb_graphormer_lite_full_run_exp
DATA_CACHE: /Users/pelmeshek1706/Desktop/projects/airest-face/output/jupyter-notebook/rafdb_mediapipe_768_publish/cache
RESULTS_DIR: /Users/pelmeshek1706/Desktop/projects/airest-face/output/jupyter-notebook/mediapipe_pyfeat_test/results


## Implementation

This cell contains the exact model/data/inference implementation used by the comparison script.
It exposes the requested single-image API and now supports explicit Graphormer device selection for timing comparisons:

```python
image_for_test, facial_landmarks_for_test = pick_image_frpm_dataset(emotion='anger')
result_gformer_mps = inferance_gformer(facial_landmarks_for_test, device='mps')
result_gformer_cpu = inferance_gformer(facial_landmarks_for_test, device='cpu')
result_pyfeat = inferance_pyfeat(image_for_test)
```

`inferance_gformer` defaults to the best available backend (`mps` on Apple Silicon, otherwise `cpu`) and uses
`gformer_m_ce_sqrtw_geom_seed42` unless you pass `model_name='gformer_s_ce_sqrtw_geom_seed42'`.


In [9]:
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

os.environ.setdefault("OMP_NUM_THREADS", "1")

import cv2
import feat
import mediapipe as mp
import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path
from torch import nn


PROJECT_ROOT = Path("/Users/pelmeshek1706/Desktop/projects/airest-face")
OUTPUT_ROOT = PROJECT_ROOT / "output" / "jupyter-notebook"
EXPERIMENT_ROOT = OUTPUT_ROOT / "emotional_expressivity"
TRAINING_NOTEBOOK = EXPERIMENT_ROOT / "rafdb_graphormer_lite_full_colab_uv_mediapipe_legacy_exp.ipynb"
PROCESSED_ROWS = EXPERIMENT_ROOT / "rafdb_mediapipe_768_publish" / "cache" / "processed_rows_full.jsonl"
IMAGE_ROOT = EXPERIMENT_ROOT
RUN_ROOT = EXPERIMENT_ROOT / "rafdb_graphormer_lite_full_run_exp"
RESULTS_DIR = EXPERIMENT_ROOT / "mediapipe_pyfeat_test" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_LABELS = ["anger", "disgust", "fear", "happiness", "sadness", "surprise", "neutral"]
TARGET_LABEL_TO_ID = {name: idx for idx, name in enumerate(TARGET_LABELS)}
ID_TO_TARGET_LABEL = {idx: name for name, idx in TARGET_LABEL_TO_ID.items()}
NUM_CLASSES = len(TARGET_LABELS)
LANDMARK_COUNT = 478
FACE_MESH = mp.solutions.face_mesh
EXPERIMENTS: list[dict[str, Any]] = []
_GRAPHORMER_DEFINITIONS_LOADED = False
_GFORMER_MODEL_CACHE: dict[tuple[str, str], torch.nn.Module] = {}
_PYFEAT_DETECTOR = None


def exec_training_notebook_cell(index: int) -> None:
    nb = json.loads(TRAINING_NOTEBOOK.read_text(encoding="utf-8"))
    source = "".join(nb["cells"][index]["source"])
    exec(compile(source, f"{TRAINING_NOTEBOOK.name}:cell{index}", "exec"), globals())


def load_graphormer_definitions() -> None:
    global _GRAPHORMER_DEFINITIONS_LOADED
    if _GRAPHORMER_DEFINITIONS_LOADED:
        return
    # Cell 14 builds the exact graph topology used by the experiment.
    exec_training_notebook_cell(14)
    # Cell 20 defines GraphormerLiteClassifier. EXPERIMENTS is intentionally empty
    # so the cell's parameter-count preview loop is a no-op.
    exec_training_notebook_cell(20)
    _GRAPHORMER_DEFINITIONS_LOADED = True


def _graphormer_device_available(device: torch.device) -> bool:
    if device.type == "cpu":
        return True
    if device.type == "mps":
        return bool(torch.backends.mps.is_available())
    if device.type == "cuda":
        return bool(torch.cuda.is_available())
    return False


def normalize_graphormer_device(device: str | torch.device | None = None) -> torch.device:
    if device is None:
        if torch.backends.mps.is_available():
            return torch.device("mps")
        if torch.cuda.is_available():
            return torch.device("cuda")
        return torch.device("cpu")
    normalized = torch.device(device)
    if not _graphormer_device_available(normalized):
        raise RuntimeError(f"Requested device {normalized} is not available in this environment")
    return normalized


def available_graphormer_devices(include_cpu: bool = True) -> list[torch.device]:
    devices: list[torch.device] = []
    if include_cpu:
        devices.append(torch.device("cpu"))
    if torch.backends.mps.is_available():
        devices.append(torch.device("mps"))
    elif torch.cuda.is_available():
        devices.append(torch.device("cuda"))
    return devices


def graphormer_device_name(device: str | torch.device | None = None) -> str:
    return str(normalize_graphormer_device(device))


def _model_device(model: torch.nn.Module) -> torch.device:
    return next(model.parameters()).device


def _synchronize_device(device: torch.device) -> None:
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize(device)


def _pair_distance(batch: np.ndarray, left: int, right: int) -> np.ndarray:
    return np.linalg.norm(batch[:, left, :] - batch[:, right, :], axis=1)


def _axis_delta(batch: np.ndarray, left: int, right: int, axis: int) -> np.ndarray:
    return batch[:, left, axis] - batch[:, right, axis]


def build_engineered_geometry_features(batch: np.ndarray) -> np.ndarray:
    batch = np.asarray(batch, dtype=np.float32).reshape(-1, LANDMARK_COUNT, 3)
    left_eye_open = _pair_distance(batch, 159, 145)
    right_eye_open = _pair_distance(batch, 386, 374)
    left_brow_eye = _pair_distance(batch, 105, 159)
    right_brow_eye = _pair_distance(batch, 334, 386)
    left_cheek_mouth = _pair_distance(batch, 205, 61)
    right_cheek_mouth = _pair_distance(batch, 425, 291)
    features = [
        _pair_distance(batch, 61, 291),
        _pair_distance(batch, 13, 14),
        _pair_distance(batch, 0, 17),
        left_eye_open,
        right_eye_open,
        np.abs(left_eye_open - right_eye_open),
        left_brow_eye,
        right_brow_eye,
        np.abs(left_brow_eye - right_brow_eye),
        _pair_distance(batch, 105, 334),
        _pair_distance(batch, 1, 13),
        _pair_distance(batch, 152, 17),
        left_cheek_mouth,
        right_cheek_mouth,
        np.abs(left_cheek_mouth - right_cheek_mouth),
        np.abs(_axis_delta(batch, 61, 291, 1)),
        np.abs(_axis_delta(batch, 61, 291, 2)),
        np.abs(batch[:, 61, 1] - batch[:, 291, 1]) + np.abs(batch[:, 61, 2] - batch[:, 291, 2]),
        batch[:, list(LEFT_EYE_CORNERS), 0].mean(axis=1),
        batch[:, list(RIGHT_EYE_CORNERS), 0].mean(axis=1),
        batch[:, :, 2].std(axis=1),
    ]
    return np.stack(features, axis=1).astype(np.float32)


def latest_completed_run_dir(run_name: str) -> Path:
    candidates = sorted(
        path for path in (RUN_ROOT / "experiments").iterdir()
        if path.is_dir() and path.name.endswith(run_name) and (path / "summary.json").exists()
    )
    if not candidates:
        raise FileNotFoundError(f"No completed run found for {run_name}")
    return candidates[-1]


def load_graphormer_model(
    run_name: str,
    device: str | torch.device | None = None,
) -> torch.nn.Module:
    load_graphormer_definitions()
    normalized_device = normalize_graphormer_device(device)
    cache_key = (run_name, str(normalized_device))
    if cache_key in _GFORMER_MODEL_CACHE:
        return _GFORMER_MODEL_CACHE[cache_key]
    run_dir = latest_completed_run_dir(run_name)
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    checkpoint = torch.load(run_dir / "best_model.pt", map_location="cpu")
    model = GraphormerLiteClassifier(topology=topology, **config["model"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(normalized_device)
    model.eval()
    _GFORMER_MODEL_CACHE[cache_key] = model
    return model


def iter_processed_rows() -> Any:
    with PROCESSED_ROWS.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)


def sample_rows_per_class(seed: int = 42, per_class: int = 5, split: str = "test") -> list[dict[str, Any]]:
    rng = random.Random(seed)
    by_label: dict[str, list[dict[str, Any]]] = {label: [] for label in TARGET_LABELS}
    for row in iter_processed_rows():
        label = row.get("label_name")
        if row.get("split") != split or label not in by_label:
            continue
        if not row.get("landmark_success"):
            continue
        image_rel = row.get("_image_path")
        if not image_rel:
            image_rel = f"rafdb_mediapipe_768_publish/cache/images_full/{row['sample_id']}.jpg"
        image_path = IMAGE_ROOT / image_rel
        landmarks = row.get("landmarks_stable_eye_norm") or []
        if image_path.exists() and len(landmarks) == LANDMARK_COUNT:
            row = dict(row)
            row["image_path"] = str(image_path)
            by_label[label].append(row)

    selected = []
    for label in TARGET_LABELS:
        candidates = by_label[label]
        if len(candidates) < per_class:
            raise ValueError(f"Only {len(candidates)} candidates for label={label}")
        rng.shuffle(candidates)
        selected.extend(candidates[:per_class])
    return selected


def pick_image_from_dataset(emotion: str = "anger", split: str = "test", seed: int = 42, index: int = 0) -> tuple[str, np.ndarray]:
    """Return one cached image path and its matching normalized MediaPipe landmarks."""
    emotion = str(emotion).lower()
    if emotion not in TARGET_LABEL_TO_ID:
        raise ValueError(f"Unknown emotion={emotion!r}; expected one of {TARGET_LABELS}")
    rows = sample_rows_per_class(seed=seed, per_class=max(index + 1, 1), split=split)
    emotion_rows = [row for row in rows if row["label_name"] == emotion]
    if index >= len(emotion_rows):
        raise IndexError(f"Only {len(emotion_rows)} sampled rows available for emotion={emotion!r}")
    row = emotion_rows[index]
    landmarks = np.asarray(row["landmarks_stable_eye_norm"], dtype=np.float32).reshape(LANDMARK_COUNT, 3)
    return row["image_path"], landmarks


# Keep the misspelled alias requested in the prompt for copy/paste compatibility.
def pick_image_frpm_dataset(emotion: str = "anger", split: str = "test", seed: int = 42, index: int = 0) -> tuple[str, np.ndarray]:
    return pick_image_from_dataset(emotion=emotion, split=split, seed=seed, index=index)


@torch.no_grad()
def predict_graphormer(
    model: torch.nn.Module,
    landmarks: np.ndarray,
    device: str | torch.device | None = None,
) -> np.ndarray:
    model_device = normalize_graphormer_device(device) if device is not None else _model_device(model)
    x = torch.from_numpy(landmarks[None, :, :].astype(np.float32)).to(model_device)
    geometry = torch.from_numpy(build_engineered_geometry_features(landmarks[None, :, :])).to(model_device)
    _synchronize_device(model_device)
    logits = model(x, geometry=geometry)
    _synchronize_device(model_device)
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]
    return probs.astype(np.float64)


def inferance_gformer(
    facial_landmarks_for_test: np.ndarray,
    model_name: str = "gformer_m_ce_sqrtw_geom_seed42",
    device: str | torch.device | None = None,
) -> dict[str, Any]:
    """Run one Graphormer-lite checkpoint on normalized MediaPipe landmarks."""
    normalized_device = normalize_graphormer_device(device)
    model = load_graphormer_model(model_name, device=normalized_device)
    landmarks = np.asarray(facial_landmarks_for_test, dtype=np.float32).reshape(LANDMARK_COUNT, 3)
    probs = predict_graphormer(model, landmarks, device=normalized_device)
    label_id = int(probs.argmax())
    return {
        "model": model_name,
        "device": str(normalized_device),
        "label": TARGET_LABELS[label_id],
        "label_id": label_id,
        "probabilities": {label: float(probs[i]) for i, label in enumerate(TARGET_LABELS)},
    }


def inference_gformer(
    facial_landmarks_for_test: np.ndarray,
    model_name: str = "gformer_m_ce_sqrtw_geom_seed42",
    device: str | torch.device | None = None,
) -> dict[str, Any]:
    return inferance_gformer(facial_landmarks_for_test, model_name=model_name, device=device)


def predict_pyfeat(detector: Any, image_path: str) -> tuple[np.ndarray, float]:
    image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    faces = detector.detect_faces(image_rgb, threshold=0.95)
    threshold_used = 0.95
    if not faces or not faces[0]:
        faces = detector.detect_faces(image_rgb, threshold=0.5)
        threshold_used = 0.5
    if not faces or not faces[0]:
        raise RuntimeError("py-feat detected no face")
    landmarks = detector.detect_landmarks(image_rgb, detected_faces=faces)
    emotions = detector.detect_emotions(image_rgb, faces, landmarks)
    probs = np.asarray(emotions[0][0], dtype=np.float64)
    probs = probs / max(float(probs.sum()), 1e-12)
    return probs, threshold_used


def get_pyfeat_detector() -> Any:
    global _PYFEAT_DETECTOR
    if _PYFEAT_DETECTOR is None:
        _PYFEAT_DETECTOR = feat.Detector(device="cpu", n_jobs=1)
    return _PYFEAT_DETECTOR


def _image_input_to_path_or_array(image_for_test: str | Path | np.ndarray) -> str | np.ndarray:
    if isinstance(image_for_test, (str, Path)):
        return str(image_for_test)
    image = np.asarray(image_for_test)
    if image.ndim != 3 or image.shape[2] != 3:
        raise ValueError(f"Expected RGB/BGR image array with shape HxWx3, got {image.shape}")
    return image


def inferance_pyfeat(image_for_test: str | Path | np.ndarray) -> dict[str, Any]:
    """Run py-feat on one image path or image array and return 7 emotion probabilities."""
    detector = get_pyfeat_detector()
    image_value = _image_input_to_path_or_array(image_for_test)
    if isinstance(image_value, str):
        probs, threshold_used = predict_pyfeat(detector, image_value)
    else:
        # Accept RGB arrays. py-feat receives RGB in the rest of this notebook.
        faces = detector.detect_faces(image_value, threshold=0.95)
        threshold_used = 0.95
        if not faces or not faces[0]:
            faces = detector.detect_faces(image_value, threshold=0.5)
            threshold_used = 0.5
        if not faces or not faces[0]:
            raise RuntimeError("py-feat detected no face")
        landmarks = detector.detect_landmarks(image_value, detected_faces=faces)
        emotions = detector.detect_emotions(image_value, faces, landmarks)
        probs = np.asarray(emotions[0][0], dtype=np.float64)
        probs = probs / max(float(probs.sum()), 1e-12)
    label_id = int(probs.argmax())
    return {
        "model": "py-feat",
        "label": TARGET_LABELS[label_id],
        "label_id": label_id,
        "threshold_used": float(threshold_used),
        "probabilities": {label: float(probs[i]) for i, label in enumerate(TARGET_LABELS)},
    }


def inference_pyfeat(image_for_test: str | Path | np.ndarray) -> dict[str, Any]:
    return inferance_pyfeat(image_for_test)


def main() -> None:
    load_graphormer_definitions()
    selected_rows = sample_rows_per_class(seed=42, per_class=45, split="test")
    selected_frame = pd.DataFrame(
        [
            {
                "sample_id": row["sample_id"],
                "label": row["label_name"],
                "image_path": row["image_path"],
            }
            for row in selected_rows
        ]
    )
    selected_frame.to_csv(RESULTS_DIR / "selected_samples.csv", index=False)

    graphormer_device = normalize_graphormer_device(None)
    small_model = load_graphormer_model("gformer_s_ce_sqrtw_geom_seed42", device=graphormer_device)
    medium_model = load_graphormer_model("gformer_m_ce_sqrtw_geom_seed42", device=graphormer_device)

    print("Loading py-feat detector...")
    start = time.perf_counter()
    detector = feat.Detector(device="cpu", n_jobs=1)
    print(f"py-feat detector loaded in {time.perf_counter() - start:.1f}s")

    records: list[dict[str, Any]] = []
    for index, row in enumerate(selected_rows, start=1):
        landmarks = np.asarray(row["landmarks_stable_eye_norm"], dtype=np.float32).reshape(LANDMARK_COUNT, 3)
        pyfeat_probs, threshold_used = predict_pyfeat(detector, row["image_path"])
        small_probs = predict_graphormer(small_model, landmarks, device=graphormer_device)
        medium_probs = predict_graphormer(medium_model, landmarks, device=graphormer_device)

        record: dict[str, Any] = {
            "sample_id": row["sample_id"],
            "label": row["label_name"],
            "image_path": row["image_path"],
            "gformer_device": str(graphormer_device),
            "pyfeat_threshold_used": threshold_used,
            "pyfeat_top": TARGET_LABELS[int(pyfeat_probs.argmax())],
            "gformer_s_top": TARGET_LABELS[int(small_probs.argmax())],
            "gformer_m_top": TARGET_LABELS[int(medium_probs.argmax())],
        }
        for label_idx, label in enumerate(TARGET_LABELS):
            record[f"pyfeat_{label}"] = pyfeat_probs[label_idx]
            record[f"gformer_s_{label}"] = small_probs[label_idx]
            record[f"gformer_m_{label}"] = medium_probs[label_idx]
            record[f"absdiff_s_{label}"] = abs(small_probs[label_idx] - pyfeat_probs[label_idx])
            record[f"absdiff_m_{label}"] = abs(medium_probs[label_idx] - pyfeat_probs[label_idx])
            record[f"signeddiff_s_{label}"] = small_probs[label_idx] - pyfeat_probs[label_idx]
            record[f"signeddiff_m_{label}"] = medium_probs[label_idx] - pyfeat_probs[label_idx]
        records.append(record)
        print(index, row["sample_id"], row["label_name"], record["pyfeat_top"], record["gformer_s_top"], record["gformer_m_top"])

    predictions = pd.DataFrame(records)
    predictions.to_csv(RESULTS_DIR / "pyfeat_graphormer_predictions.csv", index=False)

    probability_rows = []
    for label in TARGET_LABELS:
        probability_rows.append(
            {
                "emotion": label,
                "median_pyfeat_probability": float(predictions[f"pyfeat_{label}"].median()),
                "median_gformer_s_probability": float(predictions[f"gformer_s_{label}"].median()),
                "median_gformer_m_probability": float(predictions[f"gformer_m_{label}"].median()),
                "mean_pyfeat_probability": float(predictions[f"pyfeat_{label}"].mean()),
                "mean_gformer_s_probability": float(predictions[f"gformer_s_{label}"].mean()),
                "mean_gformer_m_probability": float(predictions[f"gformer_m_{label}"].mean()),
            }
        )
    probability_frame = pd.DataFrame(probability_rows)
    probability_frame.to_csv(RESULTS_DIR / "median_probabilities_by_emotion.csv", index=False)

    median_rows = []
    for label in TARGET_LABELS:
        median_rows.append(
            {
                "emotion": label,
                "median_absdiff_gformer_s_vs_pyfeat": float(predictions[f"absdiff_s_{label}"].median()),
                "median_absdiff_gformer_m_vs_pyfeat": float(predictions[f"absdiff_m_{label}"].median()),
                "mean_absdiff_gformer_s_vs_pyfeat": float(predictions[f"absdiff_s_{label}"].mean()),
                "mean_absdiff_gformer_m_vs_pyfeat": float(predictions[f"absdiff_m_{label}"].mean()),
                "median_signeddiff_gformer_s_minus_pyfeat": float(predictions[f"signeddiff_s_{label}"].median()),
                "median_signeddiff_gformer_m_minus_pyfeat": float(predictions[f"signeddiff_m_{label}"].median()),
            }
        )
    median_frame = pd.DataFrame(median_rows)
    median_frame.to_csv(RESULTS_DIR / "median_absdiff_by_emotion.csv", index=False)

    agreement = pd.DataFrame(
        [
            {
                "comparison": "gformer_s_top_vs_pyfeat_top",
                "agreement_rate": float((predictions["gformer_s_top"] == predictions["pyfeat_top"]).mean()),
                "n": int(len(predictions)),
            },
            {
                "comparison": "gformer_m_top_vs_pyfeat_top",
                "agreement_rate": float((predictions["gformer_m_top"] == predictions["pyfeat_top"]).mean()),
                "n": int(len(predictions)),
            },
            {
                "comparison": "pyfeat_top_vs_dataset_label",
                "agreement_rate": float((predictions["pyfeat_top"] == predictions["label"]).mean()),
                "n": int(len(predictions)),
            },
            {
                "comparison": "gformer_s_top_vs_dataset_label",
                "agreement_rate": float((predictions["gformer_s_top"] == predictions["label"]).mean()),
                "n": int(len(predictions)),
            },
            {
                "comparison": "gformer_m_top_vs_dataset_label",
                "agreement_rate": float((predictions["gformer_m_top"] == predictions["label"]).mean()),
                "n": int(len(predictions)),
            },
        ]
    )
    agreement.to_csv(RESULTS_DIR / "top_label_agreement.csv", index=False)

    class_rows = []
    for true_label, group in predictions.groupby("label", sort=False):
        class_rows.append(
            {
                "true_label": true_label,
                "n": int(len(group)),
                "pyfeat_top_matches_label": float((group["pyfeat_top"] == group["label"]).mean()),
                "gformer_s_top_matches_label": float((group["gformer_s_top"] == group["label"]).mean()),
                "gformer_m_top_matches_label": float((group["gformer_m_top"] == group["label"]).mean()),
                "median_absdiff_s_all_emotions": float(
                    pd.Series(group[[f"absdiff_s_{label}" for label in TARGET_LABELS]].to_numpy().reshape(-1)).median()
                ),
                "median_absdiff_m_all_emotions": float(
                    pd.Series(group[[f"absdiff_m_{label}" for label in TARGET_LABELS]].to_numpy().reshape(-1)).median()
                ),
                "mean_absdiff_s_all_emotions": float(
                    group[[f"absdiff_s_{label}" for label in TARGET_LABELS]].to_numpy().reshape(-1).mean()
                ),
                "mean_absdiff_m_all_emotions": float(
                    group[[f"absdiff_m_{label}" for label in TARGET_LABELS]].to_numpy().reshape(-1).mean()
                ),
            }
        )
    class_frame = pd.DataFrame(class_rows)
    class_frame.to_csv(RESULTS_DIR / "class_level_comparison.csv", index=False)

    print("\nMedian probabilities by emotion:")
    print(probability_frame.to_string(index=False))
    print("\nMedian absolute difference by emotion:")
    print(median_frame.to_string(index=False))
    print("\nTop-label agreement:")
    print(agreement.to_string(index=False))
    print("\nClass-level comparison:")
    print(class_frame.to_string(index=False))
    print("\nSaved results to", RESULTS_DIR)


## Single Image Inference API

This is the direct lоб-в-лоб flow: pick one image from a specific class, run Graphormer-lite on landmarks, run `py-feat` on the raw image.

In [5]:
preferred_gformer_device = graphormer_device_name(None)
image_for_test, facial_landmarks_for_test = pick_image_frpm_dataset(emotion='happiness')

result_gformer_best = inferance_gformer(
    facial_landmarks_for_test,
    device=preferred_gformer_device,
)
result_gformer_cpu = inferance_gformer(
    facial_landmarks_for_test,
    device='cpu',
)
result_pyfeat = inferance_pyfeat(image_for_test)

print('image:', image_for_test)
print(f'Graphormer-lite result on {preferred_gformer_device}:')
display(result_gformer_best)
print('Graphormer-lite result on cpu:')
display(result_gformer_cpu)
print('py-feat result:')
display(result_pyfeat)


image: /Users/pelmeshek1706/Desktop/projects/airest-face/output/jupyter-notebook/rafdb_mediapipe_768_publish/cache/images_full/test_012023.jpg
Graphormer-lite result:


{'model': 'gformer_m_ce_sqrtw_geom_seed42',
 'label': 'happiness',
 'label_id': 3,
 'probabilities': {'anger': 0.019247930496931076,
  'disgust': 0.07864931970834732,
  'fear': 0.017723776400089264,
  'happiness': 0.8358457088470459,
  'sadness': 0.04177406430244446,
  'surprise': 0.005950917489826679,
  'neutral': 0.0008082067361101508}}

py-feat result:


{'model': 'py-feat',
 'label': 'happiness',
 'label_id': 3,
 'threshold_used': 0.95,
 'probabilities': {'anger': 0.0324078938471207,
  'disgust': 0.16671486170705985,
  'fear': 0.00019619531205204767,
  'happiness': 0.44902513388406745,
  'sadness': 0.00764555690798626,
  'surprise': 0.0032813004767102637,
  'neutral': 0.34072905786500346}}

## Run Full Comparison

Samples 5 images per class, runs all three models, and writes CSV artifacts to `RESULTS_DIR`.

In [10]:

main()

nodes: 478
edges: 4636
expression nodes: 140
max degree bucket: 45
edge types present: {'self': 478, 'mesh_local': 2660, 'same_region': 1326, 'symmetry': 30, 'action_pair': 26, 'anchor_to_region': 116, 'global_non_edge': 223848}
Loading py-feat detector...
py-feat detector loaded in 2.9s
1 test_008115 anger anger sadness anger
2 test_014402 anger neutral anger anger
3 test_002625 anger anger anger anger
4 test_012842 anger neutral anger sadness
5 test_002763 anger neutral sadness anger
6 test_013155 anger anger anger anger
7 test_012987 anger neutral anger anger
8 test_014560 anger neutral anger anger
9 test_012944 anger sadness sadness sadness
10 test_014116 anger surprise surprise surprise
11 test_013903 anger neutral anger anger
12 test_014593 anger neutral anger anger
13 test_014710 anger neutral sadness anger
14 test_013316 anger neutral anger anger
15 test_014063 anger neutral anger anger
16 test_002821 anger anger anger anger
17 test_013465 anger neutral anger anger
18 test_0134

## Tables

The first table contains the missing absolute `py-feat` probabilities, not only the delta against `py-feat`.
The second table keeps the previous median absolute probability difference vs `py-feat`.

In [ ]:

import pandas as pd

median_probs = pd.read_csv(RESULTS_DIR / 'median_probabilities_by_emotion.csv')
median_absdiff = pd.read_csv(RESULTS_DIR / 'median_absdiff_by_emotion.csv')
agreement = pd.read_csv(RESULTS_DIR / 'top_label_agreement.csv')
class_level = pd.read_csv(RESULTS_DIR / 'class_level_comparison.csv')
selected = pd.read_csv(RESULTS_DIR / 'selected_samples.csv')
preds = pd.read_csv(RESULTS_DIR / 'pyfeat_graphormer_predictions.csv')

print('Median/mean probabilities by emotion:')
display(median_probs)

print('Median absolute probability difference vs py-feat:')
display(median_absdiff)

print('Top-label agreement:')
display(agreement)

print('Class-level comparison:')
display(class_level)

print('Selected samples:')
display(selected[['sample_id', 'label_name', 'image_path']])

print('Per-image top labels:')
display(preds[['sample_id', 'label_name', 'pyfeat_top', 'gformer_s_top', 'gformer_m_top']])

## Model Footprint And Latency

This section benchmarks the deployed inference footprint for the three models used in the comparison.
Graphormer-lite is measured on every available backend in the current session:

- `cpu`
- `mps` on Apple Silicon when PyTorch Metal is available
- `cuda` on systems with NVIDIA CUDA instead of MPS

The checkpoints compared are:

- `gformer_s_ce_sqrtw_geom_seed42`
- `gformer_m_ce_sqrtw_geom_seed42`
- `py-feat` default detector stack (`retinaface` + `mobilefacenet` + `xgb` AU + `resmasknet` emotion + `img2pose` + `facenet`)

Columns to read carefully:

- `device`: runtime backend used for inference.
- `disk_footprint_mb`: Graphormer checkpoint size on disk; for `py-feat`, approximate local default resource footprint.
- `param_buffer_memory_mb`: in-memory tensor memory for PyTorch parameters/buffers where available. This does not include Python object overhead or XGBoost/sklearn internals.
- `rss_load_delta_mb`: process RSS increase after loading the model in this notebook process. On `mps`, this can under-report total Metal memory pressure because Apple uses unified memory.
- `latency_ms_median`: median wall-clock latency per single sample.

Important: Graphormer latency is `precomputed landmarks -> logits`. `py-feat` latency is `raw image -> face detection -> landmarks -> emotion`, so these timings measure different deployment pipelines.


In [ ]:
import gc
import math
import os
import statistics
import time
from pathlib import Path

import pandas as pd
import torch

try:
    import psutil
except Exception:
    psutil = None


def _rss_mb() -> float:
    if psutil is None:
        return float("nan")
    return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)


def _bytes_to_mb(value: int | float | None) -> float | None:
    if value is None:
        return None
    return float(value) / (1024 ** 2)


def _torch_param_buffer_bytes(model: torch.nn.Module) -> int:
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return int(param_bytes + buffer_bytes)


def _trainable_params(model: torch.nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def _safe_stat_mb(path: Path) -> float | None:
    return path.stat().st_size / (1024 ** 2) if path.exists() else None


def _graphormer_checkpoint_mb(run_name: str) -> float:
    run_dir = latest_completed_run_dir(run_name)
    return _safe_stat_mb(run_dir / "best_model.pt")


def _pyfeat_resource_disk_mb() -> float | None:
    try:
        import feat as _feat
        resource_dir = Path(_feat.__file__).resolve().parent / "resources"
        names = [
            "mobilenet0.25_Final.pth",
            "mobilefacenet_model_best.pth.tar",
            "ResMaskNet_Z_resmasking_dropout1_rot30.pth",
            "img2pose_v1.pth",
            "facenet_20180402_114759_vggface2.pth",
            "all_data_Upperscalar_June30.pkl",
            "all_data_Upperpca_June30.pkl",
            "all_data_Lowerscalar_June30.pkl",
            "all_data_Lowerpca_June30.pkl",
            "all_data_Fullscalar_June30.pkl",
            "all_data_Fullpca_June30.pkl",
            "reference_3d_68_points_trans.npy",
        ]
        paths = [resource_dir / name for name in names]
        paths.extend(sorted(resource_dir.glob("July4_AU*_XGB.ubj")))
        return sum(path.stat().st_size for path in paths if path.exists()) / (1024 ** 2)
    except Exception:
        return None


def _collect_torch_module_bytes(obj, max_depth: int = 5) -> int:
    seen_objects: set[int] = set()
    seen_tensors: set[int] = set()
    total = 0

    def add_tensor(tensor: torch.Tensor) -> None:
        nonlocal total
        key = id(tensor)
        if key in seen_tensors:
            return
        seen_tensors.add(key)
        total += int(tensor.numel() * tensor.element_size())

    def visit(value, depth: int) -> None:
        if depth < 0 or value is None:
            return
        obj_id = id(value)
        if obj_id in seen_objects:
            return
        seen_objects.add(obj_id)
        if isinstance(value, torch.nn.Module):
            for tensor in value.parameters(recurse=True):
                add_tensor(tensor)
            for tensor in value.buffers(recurse=True):
                add_tensor(tensor)
            return
        if isinstance(value, dict):
            for item in value.values():
                visit(item, depth - 1)
            return
        if isinstance(value, (list, tuple, set)):
            for item in value:
                visit(item, depth - 1)
            return
        if hasattr(value, "__dict__"):
            for item in vars(value).values():
                visit(item, depth - 1)

    visit(obj, max_depth)
    return int(total)


def _benchmark_latency(fn, warmup: int, repeats: int) -> dict[str, float]:
    for _ in range(max(warmup, 0)):
        fn()
    times = []
    for _ in range(max(repeats, 1)):
        start = time.perf_counter()
        fn()
        times.append((time.perf_counter() - start) * 1000.0)
    return {
        "latency_ms_median": float(statistics.median(times)),
        "latency_ms_mean": float(statistics.mean(times)),
        "latency_ms_min": float(min(times)),
        "latency_ms_max": float(max(times)),
    }


def benchmark_model_footprint_and_latency(
    gformer_repeats: int = 50,
    pyfeat_repeats: int = 5,
    warmup: int = 3,
    graphormer_devices: list[str | torch.device] | None = None,
) -> pd.DataFrame:
    image_for_bench, landmarks_for_bench = pick_image_frpm_dataset(emotion="happiness", seed=42, index=0)
    rows = []

    if graphormer_devices is None:
        graphormer_devices = available_graphormer_devices(include_cpu=True)
    normalized_devices = [normalize_graphormer_device(device) for device in graphormer_devices]

    for display_name, run_name in [
        ("gformer_s", "gformer_s_ce_sqrtw_geom_seed42"),
        ("gformer_m", "gformer_m_ce_sqrtw_geom_seed42"),
    ]:
        for device in normalized_devices:
            gc.collect()
            rss_before = _rss_mb()
            model = load_graphormer_model(run_name, device=device)
            rss_after = _rss_mb()
            bench = _benchmark_latency(
                lambda model=model, device=device: predict_graphormer(model, landmarks_for_bench, device=device),
                warmup=warmup,
                repeats=gformer_repeats,
            )
            rows.append({
                "model": display_name,
                "device": str(device),
                "pipeline": "precomputed landmarks -> logits",
                "params": _trainable_params(model),
                "disk_footprint_mb": _graphormer_checkpoint_mb(run_name),
                "param_buffer_memory_mb": _bytes_to_mb(_torch_param_buffer_bytes(model)),
                "rss_load_delta_mb": rss_after - rss_before if not math.isnan(rss_before) and not math.isnan(rss_after) else None,
                "latency_repeats": gformer_repeats,
                **bench,
            })

    gc.collect()
    pyfeat_cached_before = globals().get("_PYFEAT_DETECTOR") is not None
    rss_before = _rss_mb()
    detector = get_pyfeat_detector()
    rss_after = _rss_mb()
    bench = _benchmark_latency(
        lambda detector=detector: predict_pyfeat(detector, image_for_bench),
        warmup=1,
        repeats=pyfeat_repeats,
    )
    rows.append({
        "model": "py-feat_default_stack",
        "device": "cpu",
        "pipeline": "raw image -> face/landmark/emotion stack",
        "params": None,
        "disk_footprint_mb": _pyfeat_resource_disk_mb(),
        "param_buffer_memory_mb": _bytes_to_mb(_collect_torch_module_bytes(detector)),
        "rss_load_delta_mb": rss_after - rss_before if not math.isnan(rss_before) and not math.isnan(rss_after) else None,
        "latency_repeats": pyfeat_repeats,
        "pyfeat_cached_before": pyfeat_cached_before,
        **bench,
    })

    frame = pd.DataFrame(rows)
    numeric_cols = [
        "disk_footprint_mb",
        "param_buffer_memory_mb",
        "rss_load_delta_mb",
        "latency_ms_median",
        "latency_ms_mean",
        "latency_ms_min",
        "latency_ms_max",
    ]
    for col in numeric_cols:
        if col in frame:
            frame[col] = frame[col].astype(float).round(3)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    frame.to_csv(RESULTS_DIR / "model_footprint_latency.csv", index=False)
    return frame.sort_values(["model", "device"], kind="stable").reset_index(drop=True)


model_footprint_latency = benchmark_model_footprint_and_latency(
    gformer_repeats=50,
    pyfeat_repeats=5,
    warmup=3,
)
try:
    display(model_footprint_latency)
except NameError:
    print(model_footprint_latency.to_string(index=False))
print("saved:", RESULTS_DIR / "model_footprint_latency.csv")


### Benchmark Notes

Re-run the benchmark cell above in your current kernel to populate fresh timing numbers.

What changes now:

- Graphormer rows include a `device` column, so you can compare `cpu` against `mps` directly on Apple Silicon.
- `inferance_gformer(..., device='mps')` uses Metal when available; `device='cpu'` forces the CPU path.
- `py-feat` remains benchmarked on `cpu`, because this notebook only compares device backends for the Graphormer checkpoints.

Saved CSV: `output/jupyter-notebook/emotional_expressivity/mediapipe_pyfeat_test/results/model_footprint_latency.csv`
